[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc3_ml/corrections/seance1_correction.ipynb)

# Séance 3.1 — Décrire, relier, comparer — les statistiques dont le ML a besoin

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir entre moyenne et médiane selon la forme de la distribution, et lire un `describe()`
- mesurer la dispersion et la concentration d'une variable
- mesurer un lien entre deux variables, et reconnaître les trois cas où le coefficient ment
- dire si un écart entre deux groupes peut venir du hasard, et lire une p-value
- compter les individus avant de commenter un résultat

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc3_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")   ## une ligne = une COMMANDE

print(cmd.shape)
cmd.head(3)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Moyenne et médiane des quantités

> **Votre mission :**
> - Calculer la moyenne et la médiane de la colonne `qte` (nombre d'unités commandées).
> - Les arrondir à 2 décimales → `moy_qte` et `med_qte`.

In [ ]:
moy_qte = round(cmd["qte"].mean(), 2)     ## 330,77 unites
med_qte = round(cmd["qte"].median(), 2)   ## 195 unites

# Meme phenomene que sur les euros : la moyenne est loin au-dessus
print(moy_qte, "unites en moyenne, mediane a", med_qte)

In [ ]:
verifier("1a - moyenne des quantites", moy_qte == 330.77, "mean()")
verifier("1b - mediane des quantites", med_qte == 195.0, "median()")

### Exercice 2 — Un seuil, puis des tranches

> **Votre mission :**
> - La direction veut offrir la livraison aux **10 % de commandes les plus grosses**. À quel montant placer le seuil ? → `seuil` (arrondi à 2 décimales)
> - Puis découper `qte` en quatre tranches avec les bornes `[0, 50, 200, 500, 20000]` et les étiquettes `["tres_petite", "petite", "moyenne", "grosse"]` → colonne `volume`.
> - Mettre l'effectif de la tranche `moyenne` dans `nb_moyenne`.

In [ ]:
# Les 10 % du HAUT commencent au quantile 0.9 : 90 % des commandes
# sont en dessous de ce montant
seuil = round(cmd["ca"].quantile(0.9), 2)   ## 0.9 et non 0.1

# 5 bornes -> 4 tranches, dans l'ordre des etiquettes
cmd["volume"] = pd.cut(cmd["qte"], bins=[0, 50, 200, 500, 20000],
                       labels=["tres_petite", "petite", "moyenne", "grosse"])
nb_moyenne = (cmd["volume"] == "moyenne").sum()

print("livraison offerte au-dela de", seuil, "euros |", nb_moyenne, "commandes moyennes")

In [ ]:
verifier("2a - seuil des 10 % du haut", seuil == 1146.28,
         "les 10 % du haut commencent au quantile 0.9, pas 0.1")
verifier("2b - tranche moyenne", nb_moyenne == 629,
         "pd.cut(colonne, bins=[...], labels=[...])")

### Exercice 3 — La concentration, version 5 %

> **Votre mission :**
> - Quelle part du chiffre d'affaires les **5 % de commandes les plus grosses** représentent-elles ?
> - En % arrondi à 1 décimale → `part_top5`.

In [ ]:
# ascending=False : les plus grosses en premier
top = cmd["ca"].sort_values(ascending=False)
n5 = int(0.05 * len(cmd))   ## 5 % de 1 955, soit 97 commandes

part_top5 = round(100 * top.head(n5).sum() / top.sum(), 1)   ## leur part du CA
print(n5, "commandes font", part_top5, "% du chiffre d'affaires")

In [ ]:
verifier("3 - part des 5 % du haut", part_top5 == 29.6,
         "triez du plus grand au plus petit, puis divisez par le total")

### Exercice 4 — Pearson contre Spearman

> **Votre mission :**
> - Afficher la matrice de corrélation de `ca`, `nart` et `qte`, et en extraire la corrélation entre `ca` et `qte` → `r_ca_qte` (3 décimales).
> - Puis calculer les deux corrélations entre `ca` et `nart` → `r_pearson` et `r_spear` (3 décimales).
> - Laquelle des deux est la plus élevée, et qu'est-ce que ça dit de la forme du nuage ?

In [ ]:
print(cmd[["ca", "nart", "qte"]].corr().round(3))   ## la matrice

r_ca_qte = round(cmd["ca"].corr(cmd["qte"]), 3)    ## 0,848 : tres lie
r_pearson = round(cmd["ca"].corr(cmd["nart"]), 3)  ## methode par defaut

# Spearman raisonne sur les RANGS : il ne demande pas que la relation
# soit droite, seulement qu'elle monte
r_spear = round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3)

# 0,382 contre 0,671 : le nuage nart/ca est COURBE, et Pearson,
# qui cherche une droite, sous-estime le lien.
print(r_ca_qte, "| Pearson", r_pearson, "| Spearman", r_spear)

In [ ]:
verifier("4a - correlation ca / qte", r_ca_qte == 0.848, "df['ca'].corr(df['qte'])")
verifier("4b - Pearson", r_pearson == 0.382, "c'est la methode par defaut")
verifier("4c - Spearman", r_spear == 0.671, "method='spearman'")

### Exercice 5 — Le poids de 1 % des lignes

> **Votre mission :**
> - Retirer les 1 % de commandes les plus grosses → `sans`.
> - Recalculer la corrélation entre `ca` et `nart` sur ce sous-ensemble → `r_sans` (3 décimales).
> - Combien de lignes avez-vous retirées, et de combien le coefficient a-t-il bougé ?

In [ ]:
# quantile(0.99) : le montant au-dessus duquel se trouvent les 1 % du haut
seuil99 = cmd["ca"].quantile(0.99)
sans = cmd.query("ca < @seuil99")   ## 20 commandes en moins seulement

r_sans = round(sans["ca"].corr(sans["nart"]), 3)   ## 0,382 -> 0,489
print(len(cmd) - len(sans), "lignes retirees | correlation :", r_sans)

# 20 lignes sur 1 955, et le coefficient prend 28 %. Un coefficient
# cite sans son nuage de points n'est pas un resultat.

In [ ]:
verifier("5 - correlation sans les extremes", r_sans == 0.489,
         "les 1 % du haut commencent au quantile 0.99")

### Exercice 6 — Belgique contre Espagne

> **Votre mission :**
> - Extraire les paniers belges dans `be` et espagnols dans `es`, et mettre la moyenne belge arrondie à 2 décimales dans `moy_be`.
> - Tester l'écart entre les deux → `p_be_es` (arrondie à 4 décimales).
> - Conclut-on à un écart au seuil de 5 % ? Mettre `True` ou `False` dans `conclut`.

In [ ]:
be = cmd.query("pays == 'Belgique'")["ca"]   ## 72 commandes
es = cmd.query("pays == 'Espagne'")["ca"]    ## 64 commandes
moy_be = round(be.mean(), 2)   ## 446,70 euros

# equal_var=False : on ne suppose pas la meme dispersion des deux cotes
p_be_es = round(stats.ttest_ind(be, es, equal_var=False).pvalue, 4)
conclut = p_be_es < 0.05   ## 0,0683 : juste au-dessus du seuil

print(len(be), "vs", len(es), "| moyenne belge", moy_be)
print("p =", p_be_es, "| on conclut a un ecart :", conclut)

# L'ecart de 157 EUR est important, mais 72 et 64 commandes ne suffisent
# pas a l'etablir. Ce n'est PAS "pas de difference" : c'est "pas assez
# de donnees pour le dire".

In [ ]:
verifier("6a - moyenne belge", moy_be == 446.7,
         "guillemets doubles a l'exterieur, simples autour du nom du pays")
verifier("6b - p-value Belgique/Espagne", p_be_es == 0.0683,
         "stats.ttest_ind(be, es, equal_var=False).pvalue")
verifier("6c - conclusion au seuil de 5 %", not conclut,
         "0,0683 est-il inferieur a 0,05 ?")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 7 — Un describe() par marché

> **Votre mission :**
> - Produire le `describe()` du `ca` **pour chaque pays**, et n'afficher que les pays d'au moins 20 commandes.
> - Quel pays a la distribution la plus resserrée ? Laquelle la plus étalée ?
> - *Nouveau :* `describe()` s'applique aussi après un `groupby` — `df.groupby('pays')['ca'].describe()`.

In [ ]:
resume = cmd.groupby("pays")["ca"].describe()   ## huit colonnes par pays

resume.query("count >= 20").round(1)   ## sous 20 commandes, on ne commente pas

# Le Royaume-Uni : mediane 300,9, max 3 161. La Belgique : mediane 349,0,
# max 1 492. L'Irlande : mediane 657,6, max 16 775. Trois marches, trois
# regimes — et une seule moyenne globale les melange tous.
# Le reflexe : lire count AVANT toute autre colonne.

### Question 8 — Robuste ou fragile ?

> **Votre mission :**
> - Faire une copie de `cmd`, y multiplier **la plus grosse commande par 10**, puis recalculer moyenne, médiane, écart-type et écart interquartile.
> - Une seule ligne modifiée sur 1 955. De combien chaque indicateur bouge-t-il ?
> - *Rappel :* `.copy()` avant de modifier, sinon vous abîmez la table d'origine.

In [ ]:
faux = cmd.copy()   ## .copy() : on n'abime pas la table d'origine
faux.loc[faux["ca"].idxmax(), "ca"] *= 10   ## UNE ligne sur 1 955

print("moyenne :", round(cmd["ca"].mean(), 2), "->", round(faux["ca"].mean(), 2))
print("mediane :", round(cmd["ca"].median(), 2), "->", round(faux["ca"].median(), 2))
print("std     :", round(cmd["ca"].std(), 2), "->", round(faux["ca"].std(), 2))

iqr = lambda s: s.quantile(0.75) - s.quantile(0.25)
print("IQR     :", round(iqr(cmd["ca"]), 2), "->", round(iqr(faux["ca"]), 2))

# La moyenne prend 13 % et l'ecart-type QUADRUPLE, pour UNE ligne sur
# 1 955. La mediane et l'IQR ne bougent pas d'un centime. C'est ca, la
# robustesse : une seule erreur de saisie suffit a fausser un rapport
# bati sur des moyennes et des ecarts-types.

### Question 9 — Le piège du « par jour »

> **Votre mission :**
> - Calculer le chiffre d'affaires total, puis le CA moyen par jour en divisant par 7 jours de semaine.
> - Recommencer en divisant par le nombre de jours **réellement présents** dans le fichier.
> - De combien vous êtes-vous trompé, en pourcentage ? Et pourquoi rien ne vous a alerté ?

In [ ]:
par_jour = cmd.groupby("jour")["ca"].sum()
print(sorted(cmd["jour"].unique()))

faux = par_jour.sum() / 7                       ## 7 jours supposes
vrai = par_jour.sum() / cmd["jour"].nunique()   ## 6 jours reels

print("en divisant par 7  :", round(faux, 2))
print("par le nombre reel :", round(vrai, 2))
print("erreur :", round(100 * (vrai - faux) / vrai, 1), "%")

# Le samedi n'existe pas dans ce fichier : l'enseigne ne traite aucune
# commande ce jour-la. Diviser par 7 sous-estime l'activite quotidienne
# de 14,3 %. Rien n'a alerte, parce qu'un zero absent ne s'affiche nulle
# part — il faut aller le CHERCHER avec nunique() ou unique().

### Question 10 — Pourquoi Pearson sous-estimait

> **Votre mission :**
> - En partie 1 : Pearson 0,382 contre Spearman 0,671 pour `ca` et `nart`, parce que la relation est **courbe**.
> - Recalculer la corrélation de Pearson sur les **logarithmes** des deux colonnes, et retracer le nuage en échelle logarithmique.
> - Comparer les trois nombres. Qu'est-ce que le logarithme a fait à la relation ?
> - *Nouveau :* `np.log(serie)`, et `loglog=True` dans `plot`.

In [ ]:
print("Pearson brut     :", round(cmd["ca"].corr(cmd["nart"]), 3))
print("Spearman         :", round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3))
# np.log redresse une relation courbe : Pearson retrouve alors sa droite
print("Pearson sur logs :", round(np.log(cmd["ca"]).corr(np.log(cmd["nart"])), 3))

cmd.plot(kind="scatter", x="nart", y="ca", alpha=0.3, loglog=True, figsize=(7, 4))
plt.title("Les memes donnees, sur une echelle logarithmique")
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()

# 0,675 sur les logs, contre 0,671 pour Spearman : les deux methodes
# tombent d'accord. Le logarithme a REDRESSE la courbe ; sur le nuage en
# echelle log, les points s'alignent. Pearson n'avait pas tort — on lui
# demandait de mesurer une droite la ou il n'y en avait pas.

### Question 11 — Le test irlandais, au bon niveau

> **Votre mission :**
> - Le test Royaume-Uni / Irlande vu en cours traitait 256 commandes comme 256 observations indépendantes.
> - Refaire l'analyse **au niveau du client** : le panier moyen de chaque client irlandais, puis de chaque client britannique.
> - Combien d'observations reste-t-il du côté irlandais ? Que devient le test ?

In [ ]:
# Une ligne par CLIENT : c'est la bonne unite d'observation
par_client = cmd.groupby(["pays", "client_id"])["ca"].mean().reset_index()

irl_cli = par_client.query("pays == 'Irlande'")["ca"]
uk_cli = par_client.query("pays == 'Royaume-Uni'")["ca"]

print("clients irlandais   :", len(irl_cli), "->", irl_cli.round(2).tolist())
print("clients britanniques:", len(uk_cli))

# Deux observations. Un test statistique sur deux points ne veut rien
# dire, et leurs paniers moyens vont du simple au triple (2 134 et 716).
# Ce que le test de la partie 1 mesurait n'etait pas un ecart entre deux
# MARCHES, c'etait le comportement de deux acheteurs.

### Question 12 — Question de synthèse

> **Votre mission :**
> - On vous demande **une phrase** pour ouvrir le rapport annuel, à la place de « le panier moyen est de 590 € ».
> - Elle doit tenir en une ligne et porter **trois** chiffres que vous aurez calculés : un indicateur de position honnête, un de dispersion, un de concentration.
> - Puis, en commentaire, dites ce que cette phrase permet de décider que l'ancienne ne permettait pas.

In [ ]:
q1, q3 = cmd["ca"].quantile(0.25), cmd["ca"].quantile(0.75)
top = cmd["ca"].sort_values(ascending=False)
part10 = 100 * top.head(int(0.10 * len(cmd))).sum() / top.sum()

print("mediane   :", round(cmd["ca"].median(), 2), "euros")
print("moitie centrale : de", round(q1, 2), "a", round(q3, 2), "euros")
print("part des 10 % du haut :", round(part10, 1), "%")

# Une phrase possible :
#
#   "La commande typique est de 356 EUR, la moitie des commandes se
#    situant entre 190 et 660 EUR ; les 10 % les plus grosses portent
#    41,3 % du chiffre d'affaires."
#
# Ce qu'elle permet et que la moyenne ne permettait pas : fixer un seuil
# de livraison gratuite (a 660 EUR on vise le quart du haut, a 590 on ne
# visait rien de defini), et mesurer une dependance — 41 % du CA repose
# sur 195 commandes, donc sur une poignee d'acheteurs. La moyenne de
# 590 EUR ne repondait a aucune de ces deux questions.